# Special Topics in Software Engineering (TSIS) - Lab 1

Welcome to the first hands-on laboratory session for the **Topici Speciale în Inginerie Software** (_Special Topics in Software Engineering_ &mdash; TSIS) course! As mentioned in the first lecture, the objective of this course is to get you up to speed on **the latest advancements** in the field of [**large language models**](https://www.ibm.com/think/topics/large-language-models) (LLMs), focusing on the impact on developing [**agentic AI**](https://agentic.ai/what-is-agentic-ai) apps.

Unlike the first generation language models (like GPT-2, FLAN-T5, BERT etc.), newer models are more than simple chatbots; they are able to **call external tools** (access files, make API calls) and **structure their output** (to output Markdown, JSON, LaTeX etc.). Combining these powerful models with the appropriate framework, a **harness**, we can turn them into self-sufficient AI agents which are able to solve practical problems.

## Running the notebook

At the lab, we will work with the [Python](https://www.python.org/) language. Since most LLM providers expose a HTTP REST API, you could in theory query them using any programming language with a HTTP client. The agentic AI framework we'll use a bit later, [LangChain](https://www.langchain.com/), has official support for Python and JavaScript/TypeScript.

Therefore, please make sure you are comfortable wih using Python for working on the labs. For the final project, it's recommended to use Python or TypeScript for the "brain" of the AI agent (although nothing prevents you from calling into tools written in other languages).

## Gaining access to a hosted Large Language Model

Since we're interested in using Large Language Models and integrating them in our software products, we will not go into the nitty-gritty details of how to develop and train language models, or how to serve them efficiently in production (ask us if you want more resources on this). Instead, we will focus on using existing frontier models as end-users, through web APIs accessed by Python clients.

For the lab, we will use [Gemini](https://cloud.google.com/vertex-ai/generative-ai/docs/learn/models) through Google's [Vertex AI express mode](https://cloud.google.com/vertex-ai/generative-ai/docs/start/api-keys), authenticated with a Google API key.


**Warning**: prompts and generated content are sent to a hosted model. Do not include personally identifiable information, secrets, or private data in your requests. Store your API key in an environment variable and never commit it to the notebook or source control. Check your institution's Google Cloud policies before using the service.


## Installing dependencies

We will use [pip](https://pip.pypa.io/en/stable/installation/) to ensure the required packages are installed locally. If you're working locally, you might want to make sure you have configured a [virtual environment](https://docs.python.org/3/library/venv.html) beforehand.

In [ ]:
%pip install --upgrade google-genai pydantic

## Setting up the Vertex AI client

We will use Google's official [Gen AI SDK for Python](https://googleapis.github.io/python-genai/) in Vertex AI express mode. Create an API key in Google Cloud Console, then set it in the `GOOGLE_API_KEY` environment variable before running the notebook:

```bash
export GOOGLE_API_KEY="your-api-key"
```

See the [Vertex AI API keys documentation](https://cloud.google.com/vertex-ai/generative-ai/docs/start/api-keys) for creating and restricting the key. Do not paste the key into a notebook cell.


In [ ]:
import os

api_key = os.environ.get("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("GOOGLE_API_KEY is not set. Export a Google API key before running this notebook.")


In [ ]:
from google import genai
from google.genai.types import HttpOptions

client = genai.Client(
    vertexai=True,
    api_key=api_key,
    http_options=HttpOptions(api_version="v1"),
)


In [ ]:
# List Gemini models available in Vertex AI for this location.
for model in client.models.list():
    print(model.name)

## Choosing the right model

For this lab, we will use `gemini-2.5-flash`, a fast general-purpose model. See the [Vertex AI Gemini model documentation](https://cloud.google.com/vertex-ai/generative-ai/docs/learn/models) for currently available models, limits, and regional availability.


## Starting a conversation

In [ ]:
from google.genai.types import GenerateContentConfig

MODEL = "gemini-2.5-flash"

response = client.models.generate_content(
    model=MODEL,
    contents="Explain what the temperature parameter does for text generation with a Large Language Model.",
    config=GenerateContentConfig(
        system_instruction="You are a helpful assistant.",
        temperature=0.2,
    ),
)

print("Response:\n", response.text)

print("\n--- Telemetry ---")
usage = response.usage_metadata
print(f"Prompt tokens: {usage.prompt_token_count}")
print(f"Completion tokens: {usage.candidates_token_count}")
print(f"Total tokens: {usage.total_token_count}")


In [ ]:
from google.genai.types import Content, GenerateContentConfig, Part

class ConversationManager:
    def __init__(self, client: genai.Client, system_prompt: str, max_history_turns: int = 3):
        self.client = client
        self.system_prompt = system_prompt
        self.history: list[Content] = []
        self.max_messages = max_history_turns * 2
        self.model = MODEL
        self.temperature = 0.2

    def chat(self, user_input: str) -> str:
        self.history.append(Content(role="user", parts=[Part.from_text(text=user_input)]))

        # Trim complete user/assistant turns if exceeding the context window.
        if len(self.history) > self.max_messages:
            self.history = self.history[-self.max_messages :]

        response = self.client.models.generate_content(
            model=self.model,
            contents=self.history,
            config=GenerateContentConfig(
                system_instruction=self.system_prompt,
                temperature=self.temperature,
            ),
        )
        reply = response.text
        self.history.append(response.candidates[0].content)
        total_tokens = response.usage_metadata.total_token_count
        print(f"[Debug] Active context messages: {len(self.history)} | Tokens: {total_tokens}")
        return reply


In [ ]:
# Let's test the sliding window
bot = ConversationManager(
    client, "You are a helpful SE tutor. Be brief.", max_history_turns=2
)
print(bot.chat("My project is written in Rust and uses Actix-web."))
print(bot.chat("What database library fits well with this stack?"))
print(bot.chat("How do I handle connection pooling there?"))
# By turn 4, turn 1 ("My project is written in Rust...") drops out of the window!
print(bot.chat("Remind me, what web framework did I say I was using?"))

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class BugTriageReport(BaseModel):
    summary: str = Field(description="Concise 1-sentence technical summary of the issue")
    severity: Literal["LOW", "MEDIUM", "HIGH", "CRITICAL"]
    affected_component: Literal["auth", "database", "frontend", "payment", "unknown"]
    reproduction_steps: list[str] = Field(description="Ordered list of steps to reproduce")
    requires_immediate_rollback: bool

raw_ticket = """
Hey team, ever since the v2.4 deploy 10 minutes ago, users in EU cannot complete checkout!
When they click 'Pay Now' with Stripe, the spinner hangs forever and the console logs a 502 Bad Gateway
on /api/v2/charge. Tested on Chrome and Firefox. We are losing orders right now!
"""

# Vertex AI asks Gemini for JSON matching the Pydantic schema and parses it back into an object.
parsed_response = client.models.generate_content(
    model=MODEL,
    contents=[
        {"role": "user", "parts": [{"text": "Extract structured bug triage data from raw incident reports.\n\n" + raw_ticket}]},
    ],
    config=GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=BugTriageReport,
        temperature=0.0,
    ),
)

triage_obj: BugTriageReport = parsed_response.parsed
print(f"Type: {type(triage_obj)}")
print(f"Severity: {triage_obj.severity} | Rollback? {triage_obj.requires_immediate_rollback}")
print(f"Component: {triage_obj.affected_component}")
print(f"Steps: {triage_obj.reproduction_steps}")


## Tool calling: the application remains in control

Tool calling is not the model executing Python. The model chooses a function and supplies arguments; **our application validates and executes the function**, then sends the result back as a tool message. The model can then produce a final answer using that observation.

In [ ]:
import json
from google.genai import types

def get_service_health(service_name: str) -> str:
    telemetry = {
        "auth-service": {"status": "healthy", "latency_ms": 42, "error_rate_percent": 0.01},
        "payment-gateway": {"status": "degraded", "latency_ms": 1850, "error_rate_percent": 14.5},
    }
    if service_name not in telemetry:
        raise ValueError(f"Unknown service: {service_name}")
    return json.dumps(telemetry[service_name])

service_health_declaration = types.FunctionDeclaration(
    name="get_service_health",
    description="Read current health telemetry for an allow-listed production service.",
    parameters={
        "type": "object",
        "properties": {
            "service_name": {
                "type": "string",
                "enum": ["auth-service", "payment-gateway"],
            }
        },
        "required": ["service_name"],
    },
)
tools = [types.Tool(function_declarations=[service_health_declaration])]

contents = [
    types.Content(role="user", parts=[types.Part.from_text(text="Why is checkout failing? Check the payment gateway and summarize the evidence.")]),
]

# Call 1: the model may request a tool, but it cannot execute it itself.
first = client.models.generate_content(
    model=MODEL,
    contents=contents,
    config=types.GenerateContentConfig(
        system_instruction="You are an SRE assistant. Inspect telemetry before explaining incidents.",
        tools=tools,
        temperature=0.2,
    ),
)
contents.append(first.candidates[0].content)

function_calls = first.function_calls
if not function_calls:
    print(first.text)
else:
    function_responses = []
    for function_call in function_calls:
        if function_call.name != "get_service_health":
            raise ValueError(f"Refusing unknown tool: {function_call.name}")
        print(f"[application] calling {function_call.name}({function_call.args})")
        observation = get_service_health(**function_call.args)
        function_responses.append(types.Part.from_function_response(
            name=function_call.name,
            response={"result": observation},
        ))
    contents.append(types.Content(role="tool", parts=function_responses))

    # Call 2: the model receives the observation and writes the user-facing answer.
    final = client.models.generate_content(
        model=MODEL,
        contents=contents,
        config=types.GenerateContentConfig(temperature=0.2, max_output_tokens=180),
    )
    print("\nFinal report:\n" + final.text)


### Exercise 2 — extend the tool safely

Add a second read-only tool, such as `get_recent_deploys()`, and expose it in the schema. Keep the implementation deterministic and allow-listed. Then ask a question that requires both tools. What happens if the model requests a tool or argument that your application does not recognize?

### Exercise 3 — mini design challenge

Design a small agent for a software engineering task (for example, a test-failure triager). Specify:

1. the system instruction and user input;
2. the structured result you would validate;
3. one or two tools, including their safety checks; and
4. one failure mode and how a human or ordinary code would detect it.

Do not implement a tool that deletes data, changes production, or sends messages.

## Takeaways

- An LLM API is a probabilistic component, not a database or an authority.
- Context is explicit: the application decides what history to send.
- Structured output gives us a machine-checkable interface, but not guaranteed truth.
- Tool calling is an application-controlled loop: request → validate → execute → observe → respond.
- Least privilege, allow-lists, timeouts, logging, and human approval matter more as tools become capable.

Before leaving, save your answers to Exercises 1–3 and delete any local copies of API keys. In the next laboratory we will build on this loop to make an agent more reliable and observable.